In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

transactions_clean = pd.read_csv(r'C:\Users\HP\Desktop\Project 2 - Cohort - Retention - CLTV\Data\processed\cleaned_transactions.csv')

transactions_clean['purchase_date'] = pd.to_datetime(transactions_clean['purchase_date'])
transactions_clean['cohort_month'] = pd.to_datetime(
    transactions_clean['cohort_month'].astype(str)
).dt.to_period('M')

print("Data Loaded ✅")
print("Shape:", transactions_clean.shape)
print(transactions_clean.head())

Data Loaded ✅
Shape: (2211, 11)
  transaction_id    user_id purchase_date  amount_inr            product  \
0     TXN_329258  USR_00001    2024-01-17      448.38  One-Time Purchase   
1     TXN_948749  USR_00002    2023-10-15     2172.81           Pro Plan   
2     TXN_201414  USR_00003    2023-06-24     5510.05    Enterprise Plan   
3     TXN_969693  USR_00004    2023-01-23      669.36  One-Time Purchase   
4     TXN_183667  USR_00005    2023-07-05      829.63         Basic Plan   

      status acquisition_channel   region   device cohort_month    month  
0  completed       Google_Search    North   mobile      2024-01  2024-01  
1  completed       Meta_Facebook  Central   mobile      2023-10  2023-10  
2  completed               Email    South  desktop      2023-06  2023-06  
3  completed             Organic  Central   tablet      2023-01  2023-01  
4  completed              TikTok    South   mobile      2023-07  2023-07  


In [2]:
# Average Order Value per cohort
aov = transactions_clean.groupby('cohort_month').agg(
    total_revenue=('amount_inr', 'sum'),
    total_orders=('transaction_id', 'count')
).reset_index()

aov['AOV'] = (aov['total_revenue'] / aov['total_orders']).round(2)

print("=== AVERAGE ORDER VALUE PER COHORT ===")
print(aov)

=== AVERAGE ORDER VALUE PER COHORT ===
   cohort_month  total_revenue  total_orders      AOV
0       2023-01      463136.37           144  3216.22
1       2023-02      553178.69           154  3592.07
2       2023-03      614020.63           186  3301.19
3       2023-04      590484.99           178  3317.33
4       2023-05      565282.02           155  3646.98
5       2023-06      583438.61           177  3296.26
6       2023-07      646411.25           224  2885.76
7       2023-08      492171.98           175  2812.41
8       2023-09      546029.39           185  2951.51
9       2023-10      719165.41           211  3408.37
10      2023-11      570510.07           186  3067.26
11      2023-12      613466.64           191  3211.87
12      2024-01       73401.78            28  2621.49
13      2024-02       36149.15            13  2780.70
14      2024-03        6224.45             3  2074.82
15      2024-04        1600.95             1  1600.95


In [3]:
# Purchase Frequency

purchase_frequency = (
    transactions_clean
    .groupby("user_id")["transaction_id"]
    .nunique()
    .reset_index()
)

purchase_frequency.columns = [
    "user_id",
    "purchase_frequency"
]

purchase_frequency.head()

,user_id,purchase_frequency
0,USR_00001,1
1,USR_00002,1
2,USR_00003,1
3,USR_00004,1
4,USR_00005,2


In [4]:
purchase_frequency.describe()

,purchase_frequency
count,1546.000000
mean,1.430142
std,0.673567
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,5.000000


In [5]:
purchase_frequency.sort_values(
    "purchase_frequency",
    ascending=False
).head(10)

,user_id,purchase_frequency
1519,USR_01966,5
537,USR_00701,4
361,USR_00474,4
1296,USR_01658,4
1511,USR_01954,4
217,USR_00278,4
999,USR_01298,4
1292,USR_01654,4
124,USR_00153,4
593,USR_00778,4


In [6]:
# Customer Revenue

customer_revenue = (
    transactions_clean
    .groupby("user_id")
    .agg(
        total_revenue=("amount_inr", "sum"),
        total_orders=("transaction_id", "nunique")
    )
    .reset_index()
)

customer_revenue.head()

,user_id,total_revenue,total_orders
0,USR_00001,448.38,1
1,USR_00002,2172.81,1
2,USR_00003,5510.05,1
3,USR_00004,669.36,1
4,USR_00005,4116.35,2


In [13]:
# Average Order Value per User

customer_revenue["average_order_value"] = (
    customer_revenue["total_revenue"]
    / customer_revenue["total_orders"]
).round(2)

customer_revenue.head()

,user_id,total_revenue,total_orders,average_order_value
0,USR_00001,448.38,1,448.38
1,USR_00002,2172.81,1,2172.81
2,USR_00003,5510.05,1,5510.05
3,USR_00004,669.36,1,669.36
4,USR_00005,4116.35,2,2058.17


In [15]:
customer_revenue.describe()

,total_revenue,total_orders,average_order_value
count,1546.000000,1546.000000,1546.000000
mean,4576.114088,1.430142,3227.155886
std,3865.901980,0.673567,2479.200651
min,308.190000,1.000000,308.190000
25%,1647.485000,1.000000,1315.270000
50%,3077.850000,1.000000,2447.500000
75%,6931.122500,2.000000,4598.522500
max,23168.010000,5.000000,9967.310000


In [17]:
customer_revenue.sort_values(
    "average_order_value",
    ascending=False
).head(10)

,user_id,total_revenue,total_orders,average_order_value
430,USR_00558,9967.31,1,9967.31
901,USR_01171,9962.76,1,9962.76
256,USR_00332,9957.99,1,9957.99
1420,USR_01841,9940.38,1,9940.38
1321,USR_01701,9931.23,1,9931.23
165,USR_00205,9894.00,1,9894.00
826,USR_01079,9860.70,1,9860.70
1052,USR_01366,9837.37,1,9837.37
1155,USR_01489,9823.26,1,9823.26
1188,USR_01530,9815.67,1,9815.67


In [21]:
# Historical CLTV

customer_revenue["CLTV"] = (
    customer_revenue["average_order_value"]
    * purchase_frequency["purchase_frequency"]
).round(2)

customer_revenue.head()

,user_id,total_revenue,total_orders,average_order_value,CLTV
0,USR_00001,448.38,1,448.38,448.38
1,USR_00002,2172.81,1,2172.81,2172.81
2,USR_00003,5510.05,1,5510.05,5510.05
3,USR_00004,669.36,1,669.36,669.36
4,USR_00005,4116.35,2,2058.17,4116.34
